# 04. Products for emergency management

Blocks 01 to 03 produced a DoD we trust to about 15 cm, `sig_slope`, the change that clears a slope-based level of detection. Block 03 also showed that this 95% level is optimistic, with about 11% of stable cells passing instead of 5%, so some of the smallest significant change here is noise. The calibrated LoD in block 03 keeps about 40% less net lowering. In this block we summarize that change the way a response team would ask about it: by burn severity, by subwatershed, and in channels versus hillslopes, and then run a 10 m screening pass that combines topographic and spectral change.

In [ ]:
import io

import grass.script as gs

from workshop_utils import gj, rounded, start

tools, session = start(
    "04",
    required=["dod_debiased", "sig_slope", "dtm_2022", "stream_distance", "burned"],
)
tools.r_mapcalc(expression="sig_lowering = if(sig_slope < 0, sig_slope, null())")
tools.r_mapcalc(expression="sig_raising = if(sig_slope > 0, sig_slope, null())")

## Burn severity

`burn_severity` comes from Sentinel-2 dNBR between 21 July and 9 September 2022, classed with the thresholds MTBS published for this fire. NBR uses the 20 m shortwave infrared band resampled to 10 m, and the thresholds come from a Landsat assessment, so treat the classes as approximate. See `data/README.md` for the details.

In [ ]:
severity_map = gj.Map(width=700)
severity_map.d_shade(shade="dtm_2022_relief", color="burn_severity@PERMANENT")
severity_map.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
severity_map.d_legend(
    raster="burn_severity@PERMANENT",
    title="Burn severity",
    at=(5, 35, 2, 5),
    fontsize=11,
    flags="b",
)
severity_map.show()

## Change by burn severity

Volumes grow with area, so compare classes by volume per hectare as well. The cells are 1 m, so the sum of a DoD in meters is a volume in cubic meters.

In [ ]:
def zonal_sums(cover, zones):
    return {
        z["zone"]: (z["zone_label"], z["sum"], z["n"])
        for z in tools.r_univar(map=cover, zones=zones, format="json")
    }


area = {
    z["zone"]: z["n"]
    for z in tools.r_univar(
        map="dod_debiased", zones="burn_severity@PERMANENT", format="json"
    )
}
lowering = zonal_sums("sig_lowering", "burn_severity@PERMANENT")
raising = zonal_sums("sig_raising", "burn_severity@PERMANENT")

print(
    f"{'class':22}{'area ha':>9}{'lowering m3':>13}{'raising m3':>12}{'net m3/ha':>11}"
)
for zone, label, _, _ in sorted((k, *v) for k, v in lowering.items()):
    hectares = area[zone] / 10_000
    low = lowering[zone][1]
    rise = raising.get(zone, (None, 0.0))[1]
    print(
        f"{label:22}{hectares:>9.1f}{rounded(low):>13,.0f}{rounded(rise):>12,.0f}"
        f"{rounded((low + rise) / hectares):>+11,.0f}"
    )

Net lowering per hectare rises with every step in burn severity. High severity ground lost about 1,000 m3 per hectare, about five times the low severity class and fourteen times the unburned to low class.

Read this as surface lowering, not eroded sediment. Burned ground also lost duff, litter, and understory, and the lidar ground filter sees a different surface after a fire, so part of the lowering is the fire itself. Net lowering per hectare is about the same on hillslopes as in channels (below), which also points to a broad drop of the surface rather than erosion alone.

## Change by subwatershed

A response team allocates by drainage. Split the region into subwatersheds of at least 20 ha with *r.watershed* and total the significant change in each. Subwatersheds cut by the edge of the region only carry the part inside it.

In [ ]:
tools.r_watershed(elevation="dtm_2022", threshold=200_000, basin="basins", flags="a")
tools.r_to_vect(input="basins", output="basins", type="area", flags="s")
tools.r_stats_zonal(base="basins", cover="sig_slope", method="sum", output="basin_net")
# Red for net lowering, blue for net raising, symmetric about zero.
stats = tools.r_univar(map="basin_net", format="json")
extreme = max(abs(stats["min"]), abs(stats["max"]))
tools.r_colors(
    map="basin_net",
    rules=io.StringIO(f"{-extreme} 178:24:43\n0 247:247:247\n{extreme} 33:102:172\n"),
)

net = sorted(
    (
        (z["zone"], z["sum"], z["n"])
        for z in tools.r_univar(map="sig_slope", zones="basins", format="json")
    ),
    key=lambda item: item[1],
)
print("Subwatersheds with the most net lowering:")
for zone, total, _ in net[:5]:
    print(f"  basin {zone:>5}: {rounded(total):>+12,.0f} m3")

In [ ]:
basin_map = gj.Map(width=700)
basin_map.d_shade(shade="dtm_2022_relief", color="basin_net")
basin_map.d_vect(map="basins", fill_color="none", color="white", width=1)
basin_map.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
basin_map.d_legend(
    raster="basin_net", title="Net change (m3)", at=(5, 40, 2, 5), flags="b"
)
basin_map.show()

## Channels and hillslopes

Block 02 kept channels out of the stable mask. Here they are the point: compare the change within 10 m of a channel with the change on the hillslopes.

In [ ]:
tools.r_mapcalc(expression="landform = if(stream_distance <= 10, 1, 2)")
tools.r_category(
    map="landform", separator=":", rules=io.StringIO("1:channel\n2:hillslope\n")
)
lowering = zonal_sums("sig_lowering", "landform")
raising = zonal_sums("sig_raising", "landform")
area = {
    z["zone"]: z["n"]
    for z in tools.r_univar(map="dod_debiased", zones="landform", format="json")
}
print(f"{'':10}{'area ha':>9}{'lowering m3':>13}{'raising m3':>12}{'net m3/ha':>11}")
for zone in (1, 2):
    label = lowering[zone][0]
    hectares = area[zone] / 10_000
    low, rise = lowering[zone][1], raising[zone][1]
    print(
        f"{label:10}{hectares:>9.1f}{rounded(low):>13,.0f}{rounded(rise):>12,.0f}"
        f"{rounded((low + rise) / hectares):>+11,.0f}"
    )

The net change per hectare is about the same in channels and on hillslopes, but channels both lost and gained more. They lost about 1,000 m3 per hectare against about 690 on the hillslopes, and gained about six times as much per hectare, where material moving off the slopes settled.

## A 10 m screening pass

*r.dem.screen* is a coarse first pass for directing field crews. It flags topographic change where the DoD exceeds a threshold and spectral change where vegetation was lost, and ranks cells with both highest. Here the spectral change is the NBR change, post minus pre, so loss is negative.

In [ ]:
with gs.RegionManager(region="focal@PERMANENT", res=10, flags="a"):
    tools.r_resamp_stats(input="dod_debiased", output="dod_10m", method="average")
    tools.r_mapcalc(expression="nbr_change = -dnbr@PERMANENT / 1000.0")
    tools.r_dem_screen(
        dod="dod_10m",
        spectral_change="nbr_change",
        output="screen",
        topo_threshold=0.25,
        spectral_threshold=-0.15,
    )
    print(tools.r_stats(input="screen", flags="lcpn").text)

Averaging to 10 m smooths the DoD, so a 10 m cell rarely moves as much as its steepest 1 m cells. At 0.5 m almost no cells pass; at 0.25 m about 5% show both topographic and spectral change, a workable list for a field crew. Spectral change alone covers most of the burn scar, so the combined class is what narrows the search.

In [ ]:
screen_map = gj.Map(width=700)
screen_map.d_shade(shade="dtm_2022_relief", color="screen")
screen_map.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
screen_map.d_legend(
    raster="screen", title="Screening", at=(5, 35, 2, 5), fontsize=11, flags="b"
)
screen_map.show()

With roads or other infrastructure as a vector map, `infrastructure=` and `hazard_output=` mark where change meets assets.

## In context

Curtis et al. (2025) describe the 2 August 2022 storm over the burn scar: rainfall rates above 200 mm/h and 3-hour totals above 100 mm, highest in Humbug Creek, followed by rill and gully erosion and debris flows that scoured channels and left fans in lower reaches. USGS post-fire debris-flow models predicted 57,000 to 3,676,000 m3 of debris-flow volume for Humbug Creek, for a design storm with a 15-minute intensity of 40 mm/h. The observed rainfall rates were well above that.

Our volumes measure something different. They cover 6.4 km2 of the watershed, they include all surface lowering above the level of detection, not only sediment that left, and they span September 2013 to fall 2022, not just the storm. The DoD tells you where the surface changed and by how much. Attributing that change to the storm, to the fire, or to sediment delivery takes the field evidence and additional dates.

Curtis, J.A., et al., 2025. 2022 McKinney rain-on-wildfire event, dissolved oxygen sags, and a fish kill on the Klamath River, California. Scientific Reports 15, 24668. https://doi.org/10.1038/s41598-025-08179-9

## Your turn

Rerun the subwatershed totals with `threshold=50_000` (5 ha) to see the change at the scale of individual hillslope gullies, or rerun the screening pass with `topo_threshold=0.5` and compare the class areas.